# NLLB-200 Sinhala Fine-tuning v4 Hardened

This playbook avoids the known Colab/Drive/PyTorch failure modes we hit: `.gdoc` script placeholders, `evaluate.py` circular imports, NumPy binary conflicts, PyTorch 2.6 `rng_state.pth` resume failures, and save/eval-step mismatches.

Updated with the latest runtime findings: CUDA OOM after evaluation is handled by resuming with batch size 4, gradient accumulation 8, less frequent evaluation, and PYTORCH_ALLOC_CONF=expandable_segments:True.


Important: after every Colab runtime restart, rerun Cells 1-5 before resuming. Cell 4 intentionally rewrites `/content/sinhala-finetune/train.py`, `dataset.py`, and `metrics.py`; this prevents stale Drive-copied scripts from causing argument errors such as `unrecognized arguments: --val-size` or `--resume <checkpoint>`.


## Cell 1 - Runtime Health Check

In [ ]:
import subprocess, sys

# Pin NumPy below 2.x before importing heavy ML libraries. If this changes an
# already-imported NumPy, restart runtime once and run again from Cell 1.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4'], check=True)

import numpy as np
try:
    _ = np.random.RandomState()
    print(f'numpy {np.__version__} OK')
except ValueError as exc:
    print('NumPy binary conflict:', exc)
    print('Restart runtime, then rerun Cell 1.')
    raise

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU.')

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 16 if vram_gb >= 20 else (8 if vram_gb >= 14 else 4)
print('GPU:', gpu)
print(f'VRAM: {vram_gb:.1f} GB')
print('CUDA:', torch.version.cuda)
print('BATCH:', BATCH)


## Cell 2 - Mount Drive and Set Paths

In [ ]:
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    print('Drive not mounted, so nothing to flush and unmount.')

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/Projects/sinhala-subtitle'
DATA_FILE = f'{DRIVE_BASE}/clean_pairs_final.tsv'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
LOCAL_SCRIPTS = '/content/sinhala-finetune'
LOCAL_DATA = '/content/clean_pairs_final.tsv'

print('DRIVE_BASE:', DRIVE_BASE)
print('DATA_FILE:', DATA_FILE)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)


## Cell 3 - Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'numpy==1.26.4',
    'transformers==4.44.2',
    'datasets==2.19.0',
    'accelerate==0.34.2',
    'sentencepiece==0.2.0',
    'sacrebleu==2.4.0',
    'pandas>=2.0.0',
]
for package in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import accelerate, datasets, numpy as np, sacrebleu, transformers
print('numpy:', np.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('accelerate:', accelerate.__version__)
print('sacrebleu:', sacrebleu.__version__)


## Cell 4 - Copy Data and Write Clean Scripts

In [ ]:
import os, shutil, pandas as pd

os.makedirs(LOCAL_SCRIPTS, exist_ok=True)
for name in os.listdir(LOCAL_SCRIPTS):
    path = os.path.join(LOCAL_SCRIPTS, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

shutil.copy2(DATA_FILE, LOCAL_DATA)
df = pd.read_csv(LOCAL_DATA, sep='\t', dtype=str).dropna()
print(f'Dataset loaded: {len(df):,} pairs')
print(df.head(3)[['source', 'target']])


In [ ]:
%%writefile /content/sinhala-finetune/dataset.py
"""Dataset helpers for NLLB-200 Sinhala fine-tuning."""
import pandas as pd
from datasets import Dataset, DatasetDict

LANG_CODES = {
    "english": "eng_Latn",
    "tamil": "tam_Taml",
    "hindi": "hin_Deva",
    "sinhala": "sin_Sinh",
}


def load_pairs(filepath, src_lang="english"):
    df = pd.read_csv(filepath, sep="\t", dtype=str).dropna()
    if {"source", "target"}.issubset(df.columns):
        df = df[["source", "target"]].copy()
    else:
        df = df.iloc[:, [1, 2]].copy()
    df.columns = ["src", "tgt"]
    df["src"] = df["src"].str.strip()
    df["tgt"] = df["tgt"].str.strip()
    df = df[(df["src"].str.len() > 0) & (df["tgt"].str.len() > 0)]
    df["src_lang"] = LANG_CODES[src_lang]
    df["tgt_lang"] = LANG_CODES["sinhala"]
    df = df.reset_index(drop=True)
    print(f"  Loaded {len(df):,} pairs ({src_lang})")
    return df


def split(df, val_size=2000):
    val_size = min(val_size, max(1, len(df) // 10))
    val_df = df.iloc[:val_size].reset_index(drop=True)
    train_df = df.iloc[val_size:].reset_index(drop=True)
    print(f"  Train: {len(train_df):,}   Val: {len(val_df):,}")
    return DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "val": Dataset.from_pandas(val_df, preserve_index=False),
    })


def make_tokenise_fn(tokenizer, max_length=128):
    def tokenise(batch):
        tokenizer.src_lang = batch["src_lang"][0]
        tokenizer.tgt_lang = LANG_CODES["sinhala"]
        model_inputs = tokenizer(batch["src"], max_length=max_length, truncation=True)
        labels = tokenizer(text_target=batch["tgt"], max_length=max_length, truncation=True)
        tokenizer.src_lang = batch["src_lang"][0]
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenise


def build(filepath, tokenizer, src_lang="english", val_size=2000, max_length=128, num_proc=1, sample=None):
    df = load_pairs(filepath, src_lang)
    if sample:
        df = df.head(sample)
        print(f"  [sample] Using {len(df):,} rows")
    dd = split(df, val_size=val_size)
    tok = make_tokenise_fn(tokenizer, max_length=max_length)
    print("  Tokenising...")
    return dd.map(
        tok,
        batched=True,
        batch_size=256,
        num_proc=max(1, int(num_proc)),
        remove_columns=["src", "tgt", "src_lang", "tgt_lang"],
    )


In [ ]:
%%writefile /content/sinhala-finetune/metrics.py
"""Metrics using sacrebleu directly. No import of Hugging Face evaluate."""
import numpy as np
import sacrebleu as sb


def make_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [label.strip() for label in decoded_labels]
        bleu = sb.corpus_bleu(decoded_preds, [decoded_labels])
        chrf = sb.corpus_chrf(decoded_preds, [decoded_labels])
        return {"bleu": round(bleu.score, 4), "chrf": round(chrf.score, 4)}
    return compute_metrics


def benchmark_checkpoint(model, tokenizer, pairs, src_lang="eng_Latn", tgt_lang="sin_Sinh", max_new_tokens=128, num_beams=4, device="cpu"):
    import torch
    model.eval()
    model.to(device)
    tokenizer.src_lang = src_lang
    preds, refs = [], []
    for pair in pairs:
        inputs = tokenizer(pair["src"], return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            ids = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                no_repeat_ngram_size=3,
            )
        preds.append(tokenizer.decode(ids[0], skip_special_tokens=True).strip())
        refs.append(pair["ref"].strip())
    bleu = sb.corpus_bleu(preds, [refs])
    chrf = sb.corpus_chrf(preds, [refs])
    samples = [{"src": pairs[i]["src"], "ref": refs[i], "pred": preds[i]} for i in range(min(5, len(pairs)))]
    return {"bleu": round(bleu.score, 4), "chrf": round(chrf.score, 4), "samples": samples}


In [ ]:
%%writefile /content/sinhala-finetune/train.py
"""Hardened NLLB-200 fine-tuning runner for Colab."""
import argparse
import glob
import inspect
import json
import os
from datetime import datetime
from pathlib import Path

import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

import dataset as ds
import metrics as ev

DEFAULT_MODEL = "facebook/nllb-200-distilled-600M"
TARGET_LANG = "sin_Sinh"


def get_target_token_id(tokenizer, lang_code):
    tid = tokenizer.convert_tokens_to_ids(lang_code)
    if tid is not None and tid != tokenizer.unk_token_id:
        return tid
    tid = tokenizer.get_vocab().get(lang_code)
    if tid is not None:
        return tid
    raise ValueError(f"Could not find {lang_code} in tokenizer vocab.")


def disable_rng_state_restore(checkpoint):
    """Avoid PyTorch 2.6 weights_only failure on Trainer RNG-state files.

    This preserves model, optimizer, scheduler, and trainer-state resume. It only
    skips exact random-generator restoration, which is acceptable for continuing
    training from a trusted checkpoint when torch.load blocks rng_state.pth.
    """
    if not checkpoint:
        return
    for rng_file in glob.glob(os.path.join(checkpoint, "rng_state*.pth")):
        disabled = rng_file + ".disabled"
        if not os.path.exists(disabled):
            os.rename(rng_file, disabled)
            print(f"  Disabled RNG restore file for PyTorch 2.6 compatibility: {os.path.basename(rng_file)}")


def checkpoint_step(path):
    try:
        return int(Path(path).name.split("-")[-1])
    except Exception:
        return -1


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--model", default=DEFAULT_MODEL)
    p.add_argument("--output-dir", default="./checkpoints")
    p.add_argument("--epochs", type=float, default=3)
    p.add_argument("--batch-size", type=int, default=8)
    p.add_argument("--grad-accum", type=int, default=4)
    p.add_argument("--lr", type=float, default=5e-5)
    p.add_argument("--warmup", type=int, default=500)
    p.add_argument("--val-size", type=int, default=2000)
    p.add_argument("--max-length", type=int, default=128)
    p.add_argument("--src-lang", default="english", choices=["english", "tamil", "hindi"])
    p.add_argument("--fp16", action="store_true")
    p.add_argument("--num-proc", type=int, default=1)
    p.add_argument("--sample", type=int, default=None)
    p.add_argument("--resume", default=None, help="Path to checkpoint-NNN folder")
    p.add_argument("--save-steps", type=int, default=500)
    p.add_argument("--eval-steps", type=int, default=2500)
    p.add_argument("--logging-steps", type=int, default=100)
    p.add_argument("--disable-rng-restore", action="store_true", default=True)
    p.add_argument("--no-metrics", action="store_true", help="Disable BLEU/chrF during training eval")
    return p.parse_args()


def training_args_kwargs(args, out_dir, device):
    kwargs = {
        "output_dir": out_dir,
        "num_train_epochs": args.epochs,
        "per_device_train_batch_size": args.batch_size,
        "per_device_eval_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.grad_accum,
        "learning_rate": args.lr,
        "warmup_steps": args.warmup,
        "weight_decay": 0.01,
        "fp16": args.fp16 and device == "cuda",
        "predict_with_generate": not args.no_metrics,
        "generation_max_length": args.max_length,
        "eval_steps": args.eval_steps,
        "save_strategy": "steps",
        "save_steps": args.save_steps,
        "save_total_limit": 5,
        "load_best_model_at_end": False,
        "logging_dir": os.path.join(out_dir, "logs"),
        "logging_steps": args.logging_steps,
        "report_to": "none",
        "dataloader_num_workers": 0,
        "push_to_hub": False,
    }
    sig = inspect.signature(Seq2SeqTrainingArguments)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "steps"
    else:
        kwargs["evaluation_strategy"] = "steps"
    return kwargs


def main():
    args = parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n" + "=" * 60)
    print("  NLLB-200 Sinhala Fine-tuning")
    print("=" * 60)
    print(f"  Device  : {device}  |  FP16: {args.fp16 and device == 'cuda'}")
    print(f"  Batch   : {args.batch_size} x {args.grad_accum} = {args.batch_size * args.grad_accum} effective")
    if args.sample:
        print(f"  Sample  : {args.sample} rows")
    if args.resume:
        print(f"  Resume  : {args.resume}")
    print("=" * 60 + "\n")

    print("[1/4] Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    tokenizer.src_lang = ds.LANG_CODES[args.src_lang]
    model = AutoModelForSeq2SeqLM.from_pretrained(args.model)
    sin_id = get_target_token_id(tokenizer, TARGET_LANG)
    model.config.forced_bos_token_id = sin_id
    if model.config.decoder_start_token_id is None:
        model.config.decoder_start_token_id = tokenizer.eos_token_id
    model.generation_config.forced_bos_token_id = sin_id
    model.generation_config.max_new_tokens = args.max_length
    if model.generation_config.decoder_start_token_id is None:
        model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
    if model.generation_config.bos_token_id is None:
        model.generation_config.bos_token_id = tokenizer.bos_token_id
    print(f"  Parameters : {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
    print(f"  sin_Sinh id: {sin_id}")

    print("\n[2/4] Building dataset...")
    data = ds.build(
        filepath=args.data,
        tokenizer=tokenizer,
        src_lang=args.src_lang,
        val_size=args.val_size,
        max_length=args.max_length,
        num_proc=args.num_proc,
        sample=args.sample,
    )

    print("\n[3/4] Configuring trainer...")
    resume_ckpt = args.resume if args.resume and os.path.isdir(args.resume) else None
    if resume_ckpt:
        out_dir = str(Path(resume_ckpt).resolve().parent)
        if args.disable_rng_restore:
            disable_rng_state_restore(resume_ckpt)
    else:
        run_name = f"nllb-sinhala-{args.src_lang}-{datetime.now().strftime('%Y%m%d-%H%M')}"
        out_dir = os.path.join(args.output_dir, run_name)

    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100, padding=True)
    training_args = Seq2SeqTrainingArguments(**training_args_kwargs(args, out_dir, device))
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=data["train"],
        eval_dataset=data["val"],
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=None if args.no_metrics else ev.make_compute_metrics(tokenizer),
    )

    print("\n[4/4] Training...\n")
    trainer.train(resume_from_checkpoint=resume_ckpt)

    final_dir = os.path.join(out_dir, "final")
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    with open(os.path.join(out_dir, "training_summary.json"), "w", encoding="utf-8") as f:
        json.dump({"model": args.model, "src_lang": args.src_lang, "epochs": args.epochs, "final_dir": final_dir, "out_dir": out_dir}, f, indent=2)
    print("\n" + "=" * 60)
    print("  Training complete.")
    print(f"  Model saved to: {final_dir}")
    print("=" * 60 + "\n")


if __name__ == "__main__":
    main()


## Cell 5 - Verify Scripts

In [ ]:
import importlib, os, subprocess, sys

sys.path.insert(0, LOCAL_SCRIPTS)
required = {'dataset.py', 'metrics.py', 'train.py'}
actual = set(os.listdir(LOCAL_SCRIPTS))
print('Scripts:', sorted(actual))
missing = required - actual
if missing:
    raise FileNotFoundError(f'Missing scripts: {sorted(missing)}')
for forbidden in ['evaluate.py', 'evaluate.py.gdoc', 'train.py.gdoc']:
    if forbidden in actual:
        raise RuntimeError(f'Remove stale file: {forbidden}')

for module in ['dataset', 'metrics']:
    sys.modules.pop(module, None)
    importlib.import_module(module)
    print('Import OK:', module)

help_result = subprocess.run(
    [sys.executable, f'{LOCAL_SCRIPTS}/train.py', '--help'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(help_result.stdout[:2500])
if help_result.returncode != 0:
    raise RuntimeError('train.py --help failed')

required_flags = ['--val-size', '--warmup', '--resume RESUME', '--disable-rng-restore', '--no-metrics']
missing_flags = [flag for flag in required_flags if flag not in help_result.stdout]
if missing_flags:
    raise RuntimeError(
        'Stale train.py detected. Missing hardened CLI flag(s): ' + ', '.join(missing_flags)
    )

print('All scripts ready. Hardened train.py is active.')


## Cell 6 - Smoke Test

In [ ]:
import subprocess, sys

smoke_cmd = [
    sys.executable, '/content/sinhala-finetune/train.py',
    '--data', LOCAL_DATA,
    '--output-dir', '/content/smoke_test',
    '--epochs', '1',
    '--batch-size', str(BATCH),
    '--grad-accum', '1',
    '--sample', '100',
    '--save-steps', '999999',
    '--eval-steps', '999999',
    '--num-proc', '1',
    '--fp16',
    '--no-metrics',
]
print(' '.join(smoke_cmd))
result = subprocess.run(smoke_cmd, cwd=LOCAL_SCRIPTS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout[-8000:] if len(result.stdout) > 8000 else result.stdout)
if result.returncode != 0:
    raise RuntimeError(f'Smoke failed: {result.returncode}')
print('Smoke test PASSED.')


## Cell 7 - Full Training Fresh Start

In [ ]:
import os, subprocess, sys

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
train_cmd = [
    sys.executable, '/content/sinhala-finetune/train.py',
    '--data', LOCAL_DATA,
    '--output-dir', CHECKPOINT_DIR,
    '--epochs', '3',
    '--batch-size', str(BATCH),
    '--grad-accum', '4',
    '--lr', '5e-5',
    '--warmup', '500',
    '--val-size', '2000',
    '--save-steps', '500',
    '--eval-steps', '2500',
    '--logging-steps', '100',
    '--num-proc', '1',
    '--fp16',
]
print(' '.join(train_cmd))
result = subprocess.run(train_cmd, cwd=LOCAL_SCRIPTS)
if result.returncode != 0:
    raise RuntimeError(f'Training exited with code {result.returncode}')


## Cell 8 - Resume Training Safely

Run this only after Cells 1-5. It finds the latest complete checkpoint automatically and ignores incomplete checkpoint folders. It uses OOM-safe settings: batch size 4, grad accumulation 8, eval every 5000 steps.


In [ ]:
import glob, os, subprocess, sys

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

def step_num(path):
    try:
        return int(os.path.basename(os.path.normpath(path)).split('-')[-1])
    except Exception:
        return -1

required = ['trainer_state.json', 'optimizer.pt', 'scheduler.pt', 'training_args.bin']
all_ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/nllb-sinhala-*/checkpoint-*'), key=step_num)
complete = []
incomplete = []
for ckpt in all_ckpts:
    missing = [name for name in required if not os.path.exists(os.path.join(ckpt, name))]
    if missing:
        incomplete.append((ckpt, missing))
    else:
        complete.append(ckpt)

if not complete:
    raise FileNotFoundError(f'No complete checkpoints found under {CHECKPOINT_DIR}')

resume_ckpt = max(complete, key=step_num)
print('Resume from:', resume_ckpt)
if incomplete:
    print('Ignoring incomplete checkpoint folders:')
    for ckpt, missing in incomplete[-5:]:
        print(' ', ckpt, 'missing', missing)

resume_cmd = [
    sys.executable, '/content/sinhala-finetune/train.py',
    '--data', LOCAL_DATA,
    '--output-dir', CHECKPOINT_DIR,
    '--epochs', '3',
    '--batch-size', '4',
    '--grad-accum', '8',
    '--lr', '5e-5',
    '--warmup', '500',
    '--val-size', '2000',
    '--save-steps', '500',
    '--eval-steps', '5000',
    '--logging-steps', '100',
    '--num-proc', '1',
    '--fp16',
    '--resume', resume_ckpt,
]
print(' '.join(resume_cmd))
result = subprocess.run(resume_cmd, cwd=LOCAL_SCRIPTS)
if result.returncode != 0:
    raise RuntimeError(f'Resume exited with code {result.returncode}')


## Cell 9 - Emergency Resume Fix for Existing Checkpoints

Run this only if an old checkpoint still fails on `rng_state.pth`. Then rerun Cell 8.

In [ ]:
import glob, os

renamed = []
for file in glob.glob(f'{CHECKPOINT_DIR}/**/rng_state*.pth', recursive=True):
    disabled = file + '.disabled'
    if not os.path.exists(disabled):
        os.rename(file, disabled)
        renamed.append(file)
print('Disabled RNG files:', len(renamed))
for file in renamed[:20]:
    print(file)


## Copying to Another Google Account

To continue training in another account, copy only the project parent folder `sinhala-subtitle`. At minimum it must contain:

```text
sinhala-subtitle/
├── clean_pairs_final.tsv
├── NLLB200_Sinhala_v4_hardened.ipynb
├── sinhala-finetune/
│   ├── train.py
│   ├── dataset.py
│   ├── metrics.py
│   └── requirements.txt
└── checkpoints/
    └── nllb-sinhala-english-20260517-0233/
        ├── latest complete checkpoint
        └── previous complete checkpoint as backup
```

You do not need every checkpoint. Keep the latest two complete checkpoints. A complete checkpoint has `trainer_state.json`, `optimizer.pt`, `scheduler.pt`, `training_args.bin`, and model/tokenizer files. Ignore folders missing those files.


## Cell 10 - Quick Translation Test

In [ ]:
import glob, os, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

candidates = sorted(glob.glob(f'{CHECKPOINT_DIR}/**/final', recursive=True)) or sorted(glob.glob(f'{CHECKPOINT_DIR}/**/checkpoint-*', recursive=True))
if not candidates:
    raise FileNotFoundError('No model checkpoint/final folder found.')
model_path = candidates[-1]
print('Using:', model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer.src_lang = 'eng_Latn'
forced_bos_token_id = tokenizer.convert_tokens_to_ids('sin_Sinh')

sentences = ['I need to talk to you.', 'Where are you going?', 'This is not a game.']
inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id, max_new_tokens=80)
for src, pred in zip(sentences, tokenizer.batch_decode(outputs, skip_special_tokens=True)):
    print('EN:', src)
    print('SI:', pred)
    print()
